# SmolLM-135M Text Generation Demo

All three protocols: REST JSON, gRPC, REST Binary

**Important:** Run cells in order. REST JSON must run first to avoid tritonclient state issues.

In [ ]:
import os
import time
import numpy as np
import tritonclient.grpc as grpcclient
import tritonclient.http as httpclient

# =============================================================================
# Configuration
# =============================================================================
os.environ["TRITON_GRPC_URL"] = "triton-inference-server-proxy.domino-inference-dev.svc.cluster.local:50051"
os.environ["TRITON_REST_URL"] = "http://triton-inference-server-proxy.domino-inference-dev.svc.cluster.local:8080"

GRPC_URL = os.environ.get("TRITON_GRPC_URL", "localhost:50051")
REST_URL = os.environ.get("TRITON_REST_URL", "http://localhost:8080")

API_KEY = os.environ.get("DOMINO_USER_API_KEY", "")
grpc_headers = {"x-domino-api-key": API_KEY} if API_KEY else None
http_headers = {"X-Domino-Api-Key": API_KEY} if API_KEY else None

print(f"gRPC URL: {GRPC_URL}")
print(f"REST URL: {REST_URL}")
print(f"Auth: {'API Key configured' if API_KEY else 'Disabled'}")

In [ ]:
# =============================================================================
# Prepare inputs
# =============================================================================
prompt = "What is the capital of France?"
max_tokens = 128
temperature = 0.7
top_p = 0.9

print(f"Prompt: {prompt}")
print(f"Max tokens: {max_tokens}")

url = REST_URL.replace("http://", "").replace("https://", "")

In [ ]:
# =============================================================================
# 1. REST (JSON) - MUST RUN FIRST
# =============================================================================
prompt_json = np.array([prompt], dtype=np.object_)
max_tokens_json = np.array([max_tokens], dtype=np.int32)
temperature_json = np.array([temperature], dtype=np.float32)
top_p_json = np.array([top_p], dtype=np.float32)

client = httpclient.InferenceServerClient(url=url)

inp_prompt = httpclient.InferInput("prompt", [1], "BYTES")
inp_prompt.set_data_from_numpy(prompt_json, binary_data=False)
inp_max = httpclient.InferInput("max_tokens", [1], "INT32")
inp_max.set_data_from_numpy(max_tokens_json, binary_data=False)
inp_temp = httpclient.InferInput("temperature", [1], "FP32")
inp_temp.set_data_from_numpy(temperature_json, binary_data=False)
inp_top = httpclient.InferInput("top_p", [1], "FP32")
inp_top.set_data_from_numpy(top_p_json, binary_data=False)

outputs = [
    httpclient.InferRequestedOutput("generated_text", binary_data=False),
    httpclient.InferRequestedOutput("token_count", binary_data=False),
]

start = time.time()
resp = client.infer("smollm-135m-python", [inp_prompt, inp_max, inp_temp, inp_top],
                    outputs=outputs, headers=http_headers)
t_json = (time.time() - start) * 1000
text_json = resp.as_numpy("generated_text")[0]
if isinstance(text_json, bytes):
    text_json = text_json.decode("utf-8")
count_json = int(resp.as_numpy("token_count")[0])

print(f"✓ REST (JSON):   {t_json:8.2f} ms | {count_json} tokens")

In [ ]:
# =============================================================================
# 2. gRPC Protocol
# =============================================================================
prompt_grpc = np.array([prompt], dtype=np.object_)
max_tokens_grpc = np.array([max_tokens], dtype=np.int32)
temperature_grpc = np.array([temperature], dtype=np.float32)
top_p_grpc = np.array([top_p], dtype=np.float32)

client = grpcclient.InferenceServerClient(url=GRPC_URL)

inp_prompt = grpcclient.InferInput("prompt", [1], "BYTES")
inp_prompt.set_data_from_numpy(prompt_grpc)
inp_max = grpcclient.InferInput("max_tokens", [1], "INT32")
inp_max.set_data_from_numpy(max_tokens_grpc)
inp_temp = grpcclient.InferInput("temperature", [1], "FP32")
inp_temp.set_data_from_numpy(temperature_grpc)
inp_top = grpcclient.InferInput("top_p", [1], "FP32")
inp_top.set_data_from_numpy(top_p_grpc)

outputs = [
    grpcclient.InferRequestedOutput("generated_text"),
    grpcclient.InferRequestedOutput("token_count"),
]

start = time.time()
resp = client.infer("smollm-135m-python", [inp_prompt, inp_max, inp_temp, inp_top],
                    outputs=outputs, headers=grpc_headers)
t_grpc = (time.time() - start) * 1000
text_grpc = resp.as_numpy("generated_text")[0]
if isinstance(text_grpc, bytes):
    text_grpc = text_grpc.decode("utf-8")
count_grpc = int(resp.as_numpy("token_count")[0])
client.close()

print(f"✓ gRPC:          {t_grpc:8.2f} ms | {count_grpc} tokens")

In [ ]:
# =============================================================================
# 3. REST (Binary) Protocol
# =============================================================================
prompt_binary = np.array([prompt], dtype=np.object_)
max_tokens_binary = np.array([max_tokens], dtype=np.int32)
temperature_binary = np.array([temperature], dtype=np.float32)
top_p_binary = np.array([top_p], dtype=np.float32)

client = httpclient.InferenceServerClient(url=url)

inp_prompt = httpclient.InferInput("prompt", [1], "BYTES")
inp_prompt.set_data_from_numpy(prompt_binary, binary_data=False)  # BYTES always non-binary
inp_max = httpclient.InferInput("max_tokens", [1], "INT32")
inp_max.set_data_from_numpy(max_tokens_binary, binary_data=True)
inp_temp = httpclient.InferInput("temperature", [1], "FP32")
inp_temp.set_data_from_numpy(temperature_binary, binary_data=True)
inp_top = httpclient.InferInput("top_p", [1], "FP32")
inp_top.set_data_from_numpy(top_p_binary, binary_data=True)

outputs = [
    httpclient.InferRequestedOutput("generated_text", binary_data=False),
    httpclient.InferRequestedOutput("token_count", binary_data=True),
]

start = time.time()
resp = client.infer("smollm-135m-python", [inp_prompt, inp_max, inp_temp, inp_top],
                    outputs=outputs, headers=http_headers)
t_binary = (time.time() - start) * 1000
text_binary = resp.as_numpy("generated_text")[0]
if isinstance(text_binary, bytes):
    text_binary = text_binary.decode("utf-8")
count_binary = int(resp.as_numpy("token_count")[0])

print(f"✓ REST (Binary): {t_binary:8.2f} ms | {count_binary} tokens")

In [ ]:
# =============================================================================
# Results
# =============================================================================
print("Generated text (gRPC):")
print(text_grpc)